In [ ]:
import pandas as pd

DATA_URL = (
    "https://download.mlcc.google.com/"
    "mledu-datasets/california_housing_train.csv"
)

df = pd.read_csv(DATA_URL)

print(df.shape)
print(df.columns)
df.head()

(17000, 9)
Index(['longitude', 'latitude', 'housing_median_age', 'total_rooms',
       'total_bedrooms', 'population', 'households', 'median_income',
       'median_house_value'],
      dtype='object')


,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value
0,-114.31,34.19,15.0,5612.0,1283.0,1015.0,472.0,1.4936,66900.0
1,-114.47,34.40,19.0,7650.0,1901.0,1129.0,463.0,1.8200,80100.0
2,-114.56,33.69,17.0,720.0,174.0,333.0,117.0,1.6509,85700.0
3,-114.57,33.64,14.0,1501.0,337.0,515.0,226.0,3.1917,73400.0
4,-114.57,33.57,20.0,1454.0,326.0,624.0,262.0,1.9250,65500.0


In [ ]:
from numpy import random
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error

x = df.drop('median_house_value', axis=1)
y = df['median_house_value']

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state = 42)
print("Training features:", x_train.shape)
print("Testing features:", x_test.shape)
print("Training targets:", y_train.shape)
print("Testing targets:", y_test.shape)

#baseline model
baseline_model = DummyRegressor(strategy="mean")
baseline_model.fit(x_train, y_train)
baseline_mae = mean_absolute_error(y_test, baseline_model.predict(y_test))
print(f"Baseline MAE: {baseline_mae:.2f}")



Training features: (13600, 8)
Testing features: (3400, 8)
Training targets: (13600,)
Testing targets: (3400,)
Baseline MAE: 92771.89


In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

linear_model = LinearRegression()
linear_model.fit(x_train, y_train)
linear_predictions = linear_model.predict(x_test)
linear_mae = mean_absolute_error(y_test, linear_predictions)
print(f"Linear Regression MAE: {linear_mae:.2f}")

linear_rmse = np.sqrt(mean_squared_error(y_test, linear_predictions))
print(f"Linear RMSE:  ${linear_rmse:,.2f}")


Linear Regression MAE: 49983.47
Linear RMSE:  $68,078.33


In [ ]:
comparison = pd.DataFrame({"actual": y_test.iloc[:10], "predicted": linear_predictions[:10]})
comparison["error"] = (comparison["actual"] - comparison["predicted"]).abs()

comparison

,actual,predicted,error
10941,142700.0,143770.395030,1070.395030
5250,500001.0,398615.570565,101385.429435
10292,61800.0,86341.103067,24541.103067
2266,162800.0,148534.353533,14265.646467
6398,90600.0,147202.298086,56602.298086
4064,232100.0,279272.300686,47172.300686
8018,147800.0,154882.644338,7082.644338
3934,133300.0,139598.867352,6298.867352
16287,438500.0,352948.930140,85551.069860
8875,187700.0,259959.287000,72259.287000


In [ ]:
negative_count = (linear_predictions < 0).sum()
print("Negative predictions:", negative_count)

Negative predictions: 16


In [ ]:
from sklearn.ensemble import RandomForestRegressor

forest_model = RandomForestRegressor( n_estimators=200,  min_samples_leaf=2, random_state=42, n_jobs=-1)
forest_model.fit(x_train, y_train)
forest_predictions = forest_model.predict(x_test)

forest_mae = mean_absolute_error(y_test, forest_predictions)
print(f"Random Forest MAE: {forest_mae:.2f}")
forest_rmse = np.sqrt(mean_squared_error(y_test, forest_predictions))
print(f"Random Forest RMSE:  ${forest_rmse:,.2f}")

Random Forest MAE: 32133.06
Random Forest RMSE:  $49,200.03


In [ ]:
forest_train_predictions = forest_model.predict(x_train)

forest_train_mae = mean_absolute_error(y_train, forest_train_predictions)

print(f"Forest training MAE: ${forest_train_mae:,.2f}")
print(f"Forest testing MAE:  ${forest_mae:,.2f}")

Forest training MAE: $14,368.52
Forest testing MAE:  $32,133.06


In [ ]:
import pickle
import sklearn

model_artifact = { "model": forest_model, "feature_names": x_train.columns.tolist(), "sklearn_version": sklearn.__version__}

with open("housing_model.pkl", "wb") as file:
    pickle.dump(model_artifact, file)

In [ ]:
with open("housing_model.pkl", "rb") as file:
    loaded_artifact = pickle.load(file)

loaded_model = loaded_artifact["model"]
feature_names = loaded_artifact["feature_names"]

sample = x_test.iloc[[0]][feature_names]

original_prediction = forest_model.predict(sample)[0]
loaded_prediction = loaded_model.predict(sample)[0]

print("Original:", original_prediction)
print("Loaded:", loaded_prediction)

assert original_prediction == loaded_prediction

Original: 146343.5345238095
Loaded: 146343.5345238095


In [ ]:
import sys
import sklearn
import numpy
import pandas

print("Python:", sys.version.split()[0])
print("scikit-learn:", sklearn.__version__)
print("NumPy:", numpy.__version__)
print("pandas:", pandas.__version__)

Python: 3.13.15
scikit-learn: 1.6.1
NumPy: 2.1.3
pandas: 2.2.3
